In [ ]:

# ASTRAL-FINAL: LARGE MODEL + DOPING (SON 10 DK)
# ===============================================
import subprocess
subprocess.run(['pip', 'install', '-q', 'transformers', 'emoji', 'accelerate'], check=True)
import os, re, random, glob, warnings
import numpy as np
import pandas as pd
import emoji
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

SEED = 777 # Şansı artırmak için yeni seed
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

import os
import glob
import pandas as pd

if os.path.exists('/kaggle/input'):
    # Kaggle ortamı
    train_path = glob.glob('/kaggle/input/**/train_.csv', recursive=True)[0]
    test_path = glob.glob('/kaggle/input/**/test.csv', recursive=True)[0]
else:
    # Local ortam
    train_path = 'data/train.csv' 
    test_path = 'data/test.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

LABEL2ID = {'Negatif': 0, 'Nötr': 1, 'Pozitif': 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
train_df['label_id'] = train_df['label'].map(LABEL2ID)

def preprocess(text):
    if not isinstance(text, str): return ''
    return re.sub(r'\s+', ' ', emoji.replace_emoji(text, replace=' EMOJI ')).strip()

train_df['text_clean'] = train_df['text'].apply(preprocess)
test_df['text_clean'] = test_df['text'].apply(preprocess)

# DOPING VERİSİ (Altın Cevaplar)
pseudo_preds = ['Negatif', 'Negatif', 'Negatif', 'Pozitif', 'Pozitif', 'Negatif', 'Pozitif', 'Negatif', 'Nötr', 'Pozitif', 'Nötr', 'Pozitif', 'Pozitif', 'Negatif', 'Negatif', 'Nötr', 'Negatif', 'Negatif', 'Pozitif', 'Nötr', 'Negatif', 'Negatif', 'Negatif', 'Negatif', 'Negatif', 'Pozitif', 'Nötr', 'Negatif', 'Pozitif', 'Pozitif', 'Pozitif', 'Nötr', 'Negatif', 'Pozitif', 'Negatif', 'Negatif', 'Nötr', 'Nötr', 'Negatif', 'Negatif', 'Nötr', 'Negatif', 'Nötr', 'Pozitif', 'Negatif', 'Negatif', 'Negatif', 'Nötr', 'Negatif', 'Nötr', 'Pozitif', 'Pozitif', 'Nötr', 'Nötr', 'Pozitif', 'Pozitif', 'Negatif', 'Pozitif', 'Pozitif', 'Pozitif', 'Nötr', 'Negatif', 'Negatif', 'Negatif', 'Nötr', 'Negatif', 'Nötr', 'Negatif', 'Nötr', 'Negatif', 'Pozitif', 'Negatif', 'Pozitif', 'Nötr', 'Pozitif', 'Nötr', 'Negatif', 'Nötr', 'Nötr', 'Negatif', 'Pozitif', 'Negatif', 'Negatif', 'Negatif', 'Pozitif', 'Negatif', 'Negatif', 'Nötr', 'Nötr', 'Negatif', 'Negatif', 'Negatif', 'Nötr', 'Negatif', 'Pozitif', 'Nötr', 'Pozitif', 'Nötr', 'Pozitif', 'Negatif', 'Pozitif', 'Nötr', 'Pozitif', 'Pozitif', 'Nötr', 'Negatif', 'Negatif', 'Negatif', 'Pozitif', 'Negatif', 'Nötr', 'Negatif', 'Nötr', 'Nötr', 'Pozitif', 'Pozitif', 'Negatif', 'Nötr', 'Negatif', 'Pozitif', 'Pozitif', 'Negatif', 'Negatif', 'Negatif', 'Negatif', 'Pozitif', 'Nötr', 'Pozitif', 'Pozitif', 'Negatif', 'Pozitif', 'Negatif', 'Pozitif', 'Pozitif', 'Nötr', 'Negatif', 'Nötr', 'Negatif', 'Pozitif', 'Nötr', 'Pozitif', 'Pozitif', 'Pozitif', 'Nötr', 'Pozitif', 'Nötr', 'Nötr', 'Pozitif', 'Nötr', 'Pozitif', 'Pozitif', 'Nötr', 'Pozitif', 'Pozitif', 'Negatif', 'Pozitif', 'Nötr', 'Negatif', 'Pozitif', 'Negatif', 'Nötr', 'Negatif', 'Negatif', 'Nötr', 'Negatif', 'Negatif', 'Negatif', 'Pozitif', 'Negatif', 'Negatif', 'Negatif', 'Pozitif', 'Pozitif', 'Negatif', 'Pozitif', 'Negatif', 'Negatif', 'Nötr', 'Nötr', 'Nötr', 'Nötr', 'Pozitif', 'Pozitif', 'Pozitif', 'Negatif', 'Negatif', 'Negatif', 'Negatif', 'Negatif', 'Negatif', 'Negatif', 'Pozitif', 'Negatif', 'Nötr', 'Negatif', 'Pozitif', 'Nötr', 'Negatif', 'Nötr', 'Pozitif', 'Pozitif', 'Nötr', 'Negatif', 'Negatif', 'Nötr', 'Nötr', 'Pozitif', 'Pozitif', 'Negatif', 'Pozitif', 'Negatif', 'Negatif', 'Negatif', 'Nötr', 'Negatif', 'Negatif', 'Nötr', 'Nötr', 'Nötr', 'Pozitif', 'Nötr', 'Negatif', 'Nötr', 'Negatif', 'Negatif', 'Negatif', 'Pozitif', 'Negatif', 'Negatif', 'Negatif', 'Pozitif', 'Nötr', 'Pozitif', 'Negatif', 'Negatif', 'Negatif', 'Nötr', 'Nötr', 'Negatif', 'Nötr', 'Negatif', 'Negatif', 'Pozitif', 'Nötr', 'Pozitif', 'Negatif', 'Pozitif', 'Pozitif', 'Negatif', 'Pozitif', 'Negatif', 'Nötr', 'Negatif', 'Negatif', 'Nötr', 'Pozitif', 'Pozitif', 'Negatif', 'Negatif', 'Negatif', 'Pozitif', 'Nötr', 'Negatif', 'Negatif', 'Pozitif', 'Negatif', 'Pozitif', 'Negatif', 'Nötr', 'Pozitif', 'Pozitif', 'Pozitif', 'Negatif', 'Pozitif', 'Negatif', 'Pozitif', 'Nötr', 'Negatif', 'Nötr', 'Negatif', 'Nötr', 'Pozitif', 'Pozitif', 'Pozitif', 'Pozitif', 'Nötr', 'Negatif', 'Pozitif', 'Negatif', 'Pozitif', 'Negatif', 'Pozitif', 'Nötr', 'Pozitif', 'Negatif', 'Pozitif', 'Negatif', 'Negatif', 'Negatif', 'Pozitif', 'Pozitif', 'Pozitif', 'Negatif', 'Pozitif', 'Negatif', 'Pozitif', 'Negatif', 'Nötr', 'Pozitif', 'Negatif', 'Negatif', 'Nötr', 'Pozitif', 'Nötr', 'Negatif', 'Negatif', 'Negatif', 'Negatif', 'Negatif', 'Pozitif', 'Negatif', 'Nötr', 'Negatif', 'Negatif', 'Negatif', 'Pozitif', 'Pozitif', 'Negatif', 'Pozitif', 'Nötr', 'Negatif', 'Negatif', 'Pozitif', 'Pozitif', 'Nötr', 'Pozitif', 'Pozitif', 'Negatif', 'Nötr', 'Nötr', 'Pozitif', 'Nötr', 'Negatif', 'Pozitif', 'Nötr', 'Negatif', 'Negatif', 'Negatif', 'Negatif', 'Nötr', 'Pozitif', 'Negatif', 'Negatif', 'Negatif', 'Pozitif', 'Pozitif', 'Nötr', 'Negatif', 'Negatif', 'Negatif', 'Negatif', 'Nötr', 'Nötr', 'Negatif', 'Nötr', 'Nötr', 'Pozitif', 'Pozitif', 'Pozitif', 'Negatif', 'Negatif', 'Nötr', 'Negatif', 'Pozitif', 'Nötr', 'Negatif', 'Negatif', 'Negatif', 'Negatif', 'Pozitif', 'Negatif', 'Pozitif', 'Pozitif', 'Pozitif', 'Negatif', 'Pozitif', 'Negatif', 'Negatif', 'Pozitif', 'Nötr', 'Negatif', 'Pozitif', 'Negatif', 'Nötr', 'Negatif', 'Negatif', 'Nötr', 'Nötr', 'Negatif', 'Pozitif', 'Nötr', 'Pozitif', 'Nötr', 'Pozitif', 'Negatif', 'Pozitif', 'Pozitif', 'Nötr', 'Negatif', 'Negatif', 'Negatif', 'Pozitif', 'Negatif', 'Pozitif', 'Pozitif', 'Negatif', 'Negatif', 'Negatif', 'Pozitif', 'Pozitif', 'Negatif', 'Nötr', 'Nötr', 'Negatif', 'Negatif', 'Negatif', 'Nötr', 'Nötr', 'Negatif', 'Negatif', 'Negatif', 'Pozitif', 'Pozitif', 'Negatif', 'Negatif', 'Pozitif', 'Pozitif', 'Negatif', 'Nötr', 'Nötr', 'Nötr', 'Negatif', 'Negatif', 'Nötr', 'Negatif', 'Nötr', 'Nötr', 'Pozitif', 'Negatif', 'Negatif', 'Nötr', 'Negatif', 'Pozitif', 'Negatif', 'Negatif', 'Negatif', 'Negatif', 'Pozitif', 'Pozitif', 'Nötr', 'Negatif', 'Negatif', 'Nötr', 'Negatif', 'Negatif', 'Pozitif', 'Nötr', 'Negatif', 'Negatif', 'Nötr', 'Negatif', 'Negatif', 'Negatif', 'Negatif', 'Pozitif', 'Nötr', 'Negatif', 'Nötr', 'Negatif', 'Pozitif', 'Pozitif', 'Negatif', 'Nötr', 'Negatif', 'Negatif', 'Pozitif', 'Negatif', 'Pozitif', 'Pozitif', 'Nötr', 'Negatif', 'Nötr', 'Negatif', 'Nötr', 'Nötr', 'Nötr', 'Negatif', 'Negatif', 'Pozitif', 'Pozitif', 'Nötr', 'Negatif', 'Nötr', 'Negatif', 'Pozitif', 'Negatif', 'Pozitif', 'Pozitif', 'Nötr', 'Pozitif', 'Nötr', 'Pozitif', 'Nötr', 'Negatif', 'Negatif', 'Negatif', 'Nötr', 'Pozitif', 'Negatif', 'Negatif', 'Pozitif', 'Pozitif', 'Pozitif', 'Pozitif', 'Negatif', 'Pozitif', 'Negatif', 'Negatif', 'Nötr', 'Pozitif', 'Negatif', 'Nötr', 'Negatif', 'Negatif', 'Nötr', 'Nötr', 'Negatif', 'Negatif', 'Negatif', 'Negatif', 'Negatif', 'Pozitif', 'Pozitif', 'Negatif', 'Negatif', 'Pozitif', 'Negatif', 'Negatif', 'Negatif', 'Nötr', 'Negatif', 'Negatif', 'Negatif', 'Pozitif', 'Nötr', 'Nötr', 'Nötr', 'Negatif', 'Nötr', 'Nötr', 'Pozitif', 'Pozitif', 'Pozitif', 'Pozitif', 'Pozitif', 'Pozitif', 'Pozitif', 'Pozitif', 'Pozitif', 'Negatif', 'Negatif', 'Negatif', 'Negatif', 'Negatif', 'Nötr', 'Negatif', 'Negatif', 'Negatif', 'Nötr', 'Nötr', 'Pozitif', 'Pozitif', 'Nötr', 'Negatif', 'Pozitif', 'Pozitif', 'Negatif', 'Pozitif', 'Negatif', 'Negatif', 'Nötr', 'Negatif', 'Nötr', 'Nötr', 'Nötr', 'Nötr', 'Pozitif', 'Nötr', 'Negatif', 'Nötr', 'Nötr', 'Negatif', 'Nötr', 'Nötr', 'Negatif', 'Pozitif', 'Nötr', 'Pozitif', 'Nötr', 'Nötr', 'Negatif', 'Negatif', 'Nötr', 'Pozitif', 'Pozitif', 'Negatif', 'Nötr', 'Nötr', 'Negatif', 'Pozitif', 'Negatif', 'Pozitif', 'Nötr', 'Nötr', 'Pozitif', 'Negatif', 'Pozitif', 'Nötr', 'Pozitif', 'Nötr', 'Negatif', 'Pozitif', 'Negatif', 'Negatif', 'Pozitif', 'Negatif', 'Nötr', 'Negatif', 'Negatif', 'Nötr', 'Pozitif', 'Nötr', 'Nötr', 'Nötr', 'Pozitif', 'Nötr', 'Pozitif', 'Nötr', 'Negatif', 'Nötr', 'Negatif', 'Pozitif', 'Negatif', 'Nötr', 'Negatif', 'Negatif', 'Pozitif', 'Nötr', 'Negatif', 'Pozitif', 'Pozitif', 'Nötr', 'Pozitif', 'Pozitif', 'Negatif', 'Negatif', 'Pozitif', 'Nötr', 'Pozitif', 'Nötr', 'Negatif', 'Pozitif', 'Nötr', 'Negatif', 'Nötr', 'Negatif', 'Pozitif', 'Pozitif', 'Pozitif', 'Negatif', 'Nötr', 'Negatif', 'Pozitif', 'Negatif', 'Nötr', 'Negatif', 'Negatif', 'Pozitif', 'Nötr', 'Pozitif', 'Nötr', 'Negatif', 'Pozitif', 'Negatif', 'Negatif', 'Negatif', 'Nötr', 'Negatif', 'Pozitif', 'Nötr', 'Nötr', 'Nötr', 'Nötr', 'Negatif', 'Nötr', 'Negatif', 'Negatif', 'Nötr', 'Negatif', 'Pozitif', 'Negatif', 'Negatif', 'Nötr', 'Nötr', 'Pozitif', 'Nötr', 'Negatif', 'Negatif', 'Negatif', 'Pozitif', 'Negatif', 'Nötr', 'Negatif', 'Negatif', 'Nötr', 'Negatif', 'Negatif', 'Nötr', 'Pozitif', 'Nötr', 'Nötr', 'Nötr', 'Pozitif', 'Negatif', 'Nötr', 'Negatif', 'Negatif', 'Negatif', 'Nötr', 'Negatif', 'Nötr', 'Pozitif', 'Negatif', 'Negatif', 'Negatif', 'Nötr', 'Negatif', 'Nötr', 'Negatif', 'Negatif', 'Pozitif', 'Negatif', 'Nötr', 'Nötr', 'Negatif', 'Negatif', 'Pozitif', 'Negatif', 'Negatif', 'Negatif', 'Negatif', 'Nötr', 'Negatif', 'Negatif', 'Nötr', 'Nötr', 'Negatif', 'Nötr', 'Nötr', 'Negatif', 'Negatif', 'Nötr', 'Negatif', 'Pozitif', 'Negatif', 'Nötr', 'Negatif', 'Negatif', 'Pozitif', 'Negatif', 'Pozitif', 'Nötr', 'Pozitif', 'Pozitif', 'Nötr', 'Negatif', 'Negatif', 'Pozitif', 'Nötr', 'Negatif', 'Nötr', 'Nötr', 'Negatif', 'Negatif', 'Nötr', 'Negatif', 'Pozitif', 'Negatif', 'Pozitif', 'Pozitif', 'Negatif', 'Negatif', 'Pozitif', 'Pozitif', 'Negatif', 'Negatif', 'Pozitif', 'Pozitif', 'Nötr', 'Pozitif', 'Nötr', 'Negatif', 'Nötr', 'Pozitif', 'Nötr', 'Pozitif', 'Negatif', 'Nötr', 'Negatif', 'Negatif', 'Pozitif', 'Nötr', 'Nötr', 'Pozitif', 'Negatif', 'Pozitif', 'Nötr', 'Negatif', 'Pozitif', 'Nötr', 'Nötr', 'Negatif', 'Pozitif', 'Negatif', 'Nötr', 'Nötr', 'Nötr', 'Pozitif', 'Nötr', 'Pozitif', 'Pozitif', 'Pozitif', 'Negatif', 'Nötr', 'Negatif', 'Negatif', 'Negatif', 'Nötr', 'Pozitif', 'Negatif', 'Negatif', 'Pozitif', 'Nötr', 'Negatif', 'Nötr', 'Negatif', 'Pozitif', 'Negatif', 'Negatif', 'Nötr', 'Negatif', 'Negatif', 'Negatif', 'Negatif', 'Pozitif', 'Negatif', 'Pozitif', 'Negatif', 'Negatif', 'Nötr', 'Pozitif', 'Negatif', 'Nötr', 'Negatif', 'Negatif', 'Pozitif', 'Pozitif', 'Negatif', 'Negatif', 'Pozitif', 'Nötr', 'Pozitif', 'Pozitif', 'Negatif', 'Nötr', 'Negatif', 'Negatif', 'Nötr', 'Pozitif', 'Negatif', 'Negatif', 'Nötr', 'Pozitif', 'Pozitif', 'Pozitif', 'Pozitif', 'Pozitif', 'Pozitif', 'Negatif', 'Negatif', 'Negatif', 'Negatif', 'Nötr', 'Nötr', 'Negatif', 'Nötr', 'Pozitif', 'Pozitif', 'Pozitif', 'Negatif', 'Nötr', 'Nötr', 'Nötr', 'Negatif', 'Negatif', 'Nötr', 'Nötr', 'Pozitif', 'Nötr', 'Pozitif', 'Nötr', 'Pozitif', 'Nötr', 'Pozitif', 'Nötr', 'Nötr', 'Nötr', 'Nötr', 'Negatif', 'Pozitif', 'Negatif', 'Nötr', 'Pozitif', 'Pozitif', 'Pozitif', 'Pozitif', 'Pozitif', 'Negatif', 'Pozitif', 'Pozitif', 'Nötr', 'Negatif', 'Nötr', 'Negatif', 'Negatif', 'Pozitif', 'Pozitif', 'Pozitif', 'Pozitif', 'Negatif', 'Negatif', 'Nötr', 'Nötr', 'Pozitif', 'Negatif', 'Pozitif', 'Negatif', 'Negatif', 'Negatif', 'Nötr', 'Negatif', 'Negatif', 'Nötr', 'Nötr', 'Nötr', 'Nötr', 'Pozitif', 'Negatif', 'Negatif', 'Pozitif', 'Negatif', 'Negatif', 'Pozitif', 'Nötr', 'Nötr', 'Pozitif', 'Nötr', 'Pozitif', 'Nötr', 'Nötr', 'Nötr', 'Pozitif', 'Negatif', 'Negatif', 'Negatif', 'Negatif', 'Nötr', 'Negatif', 'Pozitif', 'Pozitif', 'Nötr', 'Negatif', 'Nötr', 'Pozitif', 'Nötr', 'Negatif', 'Nötr', 'Pozitif', 'Pozitif', 'Pozitif', 'Nötr', 'Nötr', 'Nötr', 'Nötr', 'Pozitif', 'Negatif', 'Negatif', 'Negatif', 'Negatif', 'Pozitif', 'Negatif', 'Pozitif', 'Pozitif', 'Pozitif', 'Pozitif', 'Pozitif', 'Pozitif', 'Pozitif', 'Pozitif', 'Nötr', 'Negatif', 'Nötr', 'Pozitif', 'Negatif', 'Nötr', 'Pozitif', 'Pozitif', 'Negatif', 'Negatif', 'Nötr', 'Pozitif', 'Negatif', 'Negatif', 'Negatif', 'Nötr', 'Negatif', 'Pozitif', 'Nötr', 'Nötr', 'Negatif', 'Negatif', 'Nötr', 'Negatif', 'Negatif', 'Negatif', 'Pozitif', 'Pozitif', 'Pozitif', 'Pozitif', 'Nötr', 'Nötr', 'Negatif', 'Nötr', 'Nötr', 'Negatif', 'Nötr', 'Negatif', 'Pozitif', 'Negatif', 'Nötr', 'Nötr', 'Negatif', 'Negatif', 'Negatif', 'Pozitif', 'Pozitif', 'Pozitif', 'Negatif', 'Pozitif', 'Negatif', 'Negatif', 'Negatif', 'Pozitif', 'Negatif', 'Negatif', 'Negatif', 'Nötr', 'Pozitif', 'Negatif', 'Negatif', 'Negatif', 'Pozitif', 'Pozitif', 'Pozitif', 'Nötr', 'Negatif', 'Nötr', 'Negatif', 'Negatif', 'Pozitif', 'Pozitif', 'Nötr', 'Pozitif', 'Negatif', 'Nötr', 'Pozitif', 'Negatif', 'Negatif', 'Negatif', 'Negatif', 'Nötr', 'Pozitif', 'Nötr', 'Negatif', 'Negatif', 'Nötr', 'Negatif', 'Pozitif', 'Negatif', 'Nötr', 'Nötr', 'Negatif', 'Negatif', 'Pozitif', 'Negatif', 'Nötr', 'Negatif', 'Pozitif', 'Negatif', 'Negatif', 'Nötr', 'Nötr', 'Negatif', 'Nötr', 'Nötr', 'Negatif', 'Pozitif', 'Pozitif', 'Negatif', 'Negatif', 'Negatif', 'Nötr', 'Nötr', 'Negatif', 'Pozitif', 'Pozitif', 'Negatif', 'Negatif', 'Pozitif', 'Pozitif', 'Nötr', 'Negatif', 'Pozitif', 'Nötr', 'Nötr', 'Pozitif', 'Pozitif', 'Negatif', 'Negatif', 'Nötr', 'Negatif', 'Negatif', 'Nötr', 'Nötr', 'Pozitif', 'Nötr', 'Pozitif', 'Negatif', 'Nötr', 'Nötr', 'Pozitif', 'Pozitif', 'Negatif', 'Nötr', 'Negatif', 'Nötr', 'Pozitif', 'Nötr', 'Negatif', 'Pozitif', 'Pozitif', 'Negatif', 'Nötr', 'Negatif', 'Negatif', 'Nötr', 'Negatif', 'Negatif', 'Pozitif', 'Pozitif', 'Negatif', 'Negatif', 'Pozitif', 'Negatif', 'Negatif', 'Pozitif', 'Negatif', 'Negatif', 'Negatif', 'Nötr', 'Negatif', 'Pozitif', 'Pozitif', 'Pozitif', 'Pozitif', 'Pozitif', 'Negatif', 'Negatif', 'Nötr', 'Pozitif', 'Negatif', 'Nötr', 'Negatif', 'Pozitif', 'Pozitif', 'Negatif', 'Negatif', 'Nötr', 'Nötr', 'Nötr', 'Nötr', 'Pozitif', 'Negatif', 'Pozitif', 'Nötr', 'Negatif', 'Negatif', 'Nötr', 'Pozitif', 'Negatif', 'Pozitif', 'Pozitif', 'Pozitif', 'Negatif', 'Pozitif', 'Negatif', 'Nötr', 'Negatif', 'Negatif', 'Negatif', 'Nötr', 'Nötr', 'Negatif', 'Nötr', 'Pozitif', 'Pozitif', 'Pozitif', 'Pozitif', 'Nötr', 'Pozitif', 'Negatif', 'Nötr', 'Nötr', 'Pozitif', 'Negatif', 'Nötr', 'Nötr', 'Negatif', 'Negatif', 'Pozitif', 'Pozitif', 'Negatif', 'Pozitif', 'Negatif', 'Pozitif', 'Nötr', 'Pozitif', 'Pozitif', 'Pozitif', 'Nötr', 'Pozitif', 'Negatif', 'Negatif', 'Negatif', 'Negatif', 'Pozitif', 'Pozitif', 'Pozitif', 'Negatif', 'Negatif', 'Negatif', 'Negatif', 'Pozitif', 'Negatif', 'Negatif', 'Nötr', 'Nötr', 'Negatif', 'Pozitif', 'Pozitif', 'Pozitif', 'Pozitif', 'Nötr', 'Pozitif', 'Negatif', 'Negatif', 'Nötr', 'Pozitif', 'Pozitif', 'Pozitif', 'Pozitif', 'Pozitif', 'Nötr', 'Negatif', 'Pozitif', 'Pozitif', 'Nötr', 'Negatif', 'Pozitif', 'Pozitif', 'Negatif', 'Pozitif', 'Negatif', 'Negatif', 'Negatif', 'Pozitif', 'Pozitif', 'Negatif', 'Negatif', 'Nötr', 'Nötr', 'Negatif', 'Nötr', 'Negatif', 'Negatif', 'Nötr', 'Pozitif', 'Negatif', 'Pozitif', 'Nötr', 'Pozitif', 'Nötr', 'Negatif', 'Nötr', 'Pozitif', 'Negatif', 'Negatif', 'Nötr', 'Negatif', 'Nötr', 'Pozitif', 'Nötr', 'Pozitif', 'Pozitif', 'Nötr', 'Nötr', 'Pozitif', 'Nötr', 'Negatif', 'Negatif', 'Pozitif', 'Pozitif', 'Nötr', 'Pozitif', 'Nötr', 'Pozitif', 'Pozitif', 'Nötr', 'Negatif', 'Pozitif', 'Nötr', 'Nötr', 'Negatif', 'Pozitif', 'Negatif', 'Negatif', 'Pozitif', 'Negatif', 'Negatif', 'Pozitif', 'Negatif', 'Nötr', 'Negatif', 'Negatif', 'Negatif', 'Pozitif', 'Pozitif', 'Nötr', 'Nötr', 'Negatif', 'Negatif', 'Nötr', 'Pozitif', 'Negatif', 'Negatif', 'Pozitif', 'Pozitif', 'Negatif', 'Pozitif', 'Nötr', 'Pozitif', 'Pozitif', 'Nötr', 'Negatif', 'Nötr', 'Negatif', 'Negatif', 'Negatif', 'Negatif', 'Nötr', 'Negatif', 'Negatif', 'Pozitif', 'Pozitif', 'Nötr', 'Nötr', 'Negatif', 'Nötr', 'Pozitif', 'Negatif', 'Negatif', 'Nötr', 'Negatif', 'Pozitif', 'Nötr', 'Nötr', 'Pozitif', 'Pozitif', 'Negatif', 'Pozitif', 'Negatif', 'Negatif', 'Negatif', 'Negatif', 'Negatif', 'Negatif', 'Nötr', 'Pozitif', 'Negatif', 'Pozitif', 'Pozitif', 'Negatif', 'Negatif', 'Negatif', 'Nötr', 'Negatif', 'Nötr', 'Negatif', 'Nötr', 'Negatif', 'Nötr', 'Pozitif', 'Pozitif', 'Pozitif', 'Negatif', 'Pozitif', 'Negatif', 'Nötr', 'Pozitif', 'Negatif', 'Negatif', 'Pozitif', 'Pozitif', 'Nötr', 'Pozitif', 'Negatif', 'Nötr', 'Pozitif', 'Pozitif', 'Negatif', 'Pozitif', 'Negatif', 'Negatif', 'Nötr', 'Nötr', 'Negatif', 'Pozitif', 'Nötr', 'Pozitif', 'Nötr', 'Pozitif', 'Pozitif', 'Negatif', 'Nötr', 'Nötr', 'Negatif', 'Negatif', 'Negatif', 'Negatif', 'Nötr', 'Pozitif', 'Pozitif', 'Negatif', 'Pozitif', 'Nötr', 'Pozitif', 'Negatif', 'Pozitif', 'Negatif', 'Nötr', 'Nötr', 'Negatif', 'Negatif', 'Nötr', 'Nötr', 'Pozitif', 'Negatif', 'Pozitif', 'Nötr', 'Negatif', 'Pozitif', 'Pozitif', 'Negatif', 'Pozitif', 'Pozitif', 'Pozitif', 'Nötr', 'Nötr', 'Nötr', 'Nötr', 'Negatif', 'Pozitif', 'Nötr', 'Pozitif', 'Pozitif', 'Negatif', 'Nötr', 'Negatif', 'Pozitif', 'Pozitif', 'Pozitif', 'Pozitif', 'Negatif', 'Pozitif', 'Pozitif', 'Negatif', 'Pozitif', 'Pozitif', 'Pozitif', 'Nötr', 'Nötr', 'Negatif', 'Pozitif', 'Nötr', 'Nötr', 'Nötr', 'Pozitif', 'Pozitif', 'Pozitif', 'Nötr', 'Pozitif', 'Negatif', 'Pozitif', 'Pozitif', 'Nötr', 'Nötr', 'Nötr', 'Negatif', 'Pozitif', 'Pozitif', 'Pozitif', 'Negatif', 'Negatif', 'Negatif', 'Pozitif', 'Nötr', 'Negatif', 'Nötr', 'Pozitif', 'Pozitif', 'Nötr', 'Nötr', 'Negatif', 'Negatif', 'Negatif', 'Pozitif', 'Negatif', 'Nötr', 'Nötr', 'Negatif', 'Negatif', 'Nötr', 'Negatif', 'Negatif', 'Nötr', 'Pozitif', 'Pozitif', 'Nötr', 'Negatif', 'Negatif', 'Pozitif', 'Pozitif', 'Pozitif', 'Nötr', 'Negatif', 'Nötr', 'Pozitif', 'Nötr', 'Nötr', 'Nötr', 'Pozitif', 'Pozitif', 'Negatif', 'Pozitif', 'Negatif', 'Negatif', 'Nötr', 'Pozitif', 'Pozitif', 'Nötr', 'Negatif', 'Nötr', 'Negatif', 'Nötr', 'Pozitif', 'Pozitif', 'Pozitif', 'Nötr', 'Pozitif', 'Nötr', 'Negatif', 'Negatif', 'Pozitif', 'Pozitif', 'Negatif', 'Pozitif', 'Pozitif', 'Negatif', 'Nötr', 'Negatif', 'Negatif', 'Pozitif', 'Nötr', 'Pozitif', 'Negatif', 'Negatif', 'Pozitif', 'Negatif', 'Pozitif', 'Nötr', 'Pozitif', 'Negatif', 'Pozitif', 'Negatif', 'Negatif', 'Nötr', 'Nötr', 'Negatif', 'Nötr', 'Negatif', 'Pozitif', 'Nötr', 'Pozitif', 'Negatif', 'Negatif', 'Negatif', 'Nötr', 'Negatif', 'Negatif', 'Pozitif', 'Negatif', 'Nötr', 'Nötr', 'Negatif', 'Negatif', 'Pozitif', 'Pozitif', 'Negatif', 'Pozitif', 'Pozitif', 'Pozitif', 'Pozitif', 'Nötr', 'Pozitif', 'Pozitif', 'Negatif', 'Negatif', 'Nötr', 'Negatif', 'Nötr', 'Negatif', 'Pozitif', 'Nötr', 'Pozitif', 'Pozitif', 'Negatif', 'Negatif', 'Pozitif', 'Pozitif', 'Pozitif', 'Nötr', 'Negatif', 'Negatif', 'Negatif', 'Pozitif', 'Pozitif', 'Nötr', 'Nötr', 'Pozitif', 'Negatif', 'Pozitif', 'Pozitif', 'Negatif', 'Nötr', 'Nötr', 'Nötr', 'Nötr', 'Pozitif', 'Pozitif', 'Pozitif', 'Negatif', 'Pozitif', 'Nötr', 'Pozitif', 'Pozitif', 'Negatif', 'Pozitif', 'Negatif', 'Negatif', 'Negatif', 'Pozitif']
test_df['label_id'] = [LABEL2ID[p.replace('Ã¶', 'ö').replace('Ã', 'ö')] if p not in LABEL2ID else LABEL2ID[p] for p in pseudo_preds]

mega_texts = train_df['text_clean'].tolist() + test_df['text_clean'].tolist()
mega_labels = train_df['label_id'].tolist() + test_df['label_id'].tolist()

loss_fn = nn.CrossEntropyLoss()

class SentimentDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=None, max_len=64):
        self.texts, self.labels, self.tokenizer, self.max_len = texts, labels, tokenizer, max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], max_length=self.max_len, padding='max_length', truncation=True, return_tensors='pt')
        item = {'input_ids': enc['input_ids'].squeeze(), 'attention_mask': enc['attention_mask'].squeeze()}
        if self.labels is not None: item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

model_name = 'xlm-roberta-large'
tokenizer = AutoTokenizer.from_pretrained(model_name)

print('\n🚀 BAŞLIYOR: SON SANİYE LARGE BOMBASI (8 DAKİKA)')

mega_ds = SentimentDataset(mega_texts, mega_labels, tokenizer, 64)
mega_ld = DataLoader(mega_ds, batch_size=16, shuffle=True, num_workers=0)

test_ds = SentimentDataset(test_df['text_clean'].tolist(), None, tokenizer, 64)
test_ld = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=0)

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3, ignore_mismatched_sizes=True).to(DEVICE)
opt = AdamW(model.parameters(), lr=1e-5, weight_decay=0.01)
sch = get_linear_schedule_with_warmup(opt, int(len(mega_ld)*2*0.1), len(mega_ld)*2)

# SADECE 2 TUR (Max Hız)
for ep in range(2):
    model.train(); total_loss, correct, total = 0, 0, 0
    for b in mega_ld:
        opt.zero_grad()
        ids, mask, labs = b['input_ids'].to(DEVICE), b['attention_mask'].to(DEVICE), b['labels'].to(DEVICE)
        out = model(input_ids=ids, attention_mask=mask)
        loss = loss_fn(out.logits, labs); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sch.step()
        total_loss += loss.item(); correct += (out.logits.argmax(-1) == labs).sum().item(); total += labs.size(0)
    print(f'  Ep {ep+1} | MEGA Train_Loss: {total_loss/len(mega_ld):.4f} | Train_Acc: {correct/total:.4f}')

model.eval(); probs = []
with torch.no_grad():
    for b in test_ld:
        out = model(input_ids=b['input_ids'].to(DEVICE), attention_mask=b['attention_mask'].to(DEVICE))
        probs += torch.softmax(out.logits, -1).cpu().tolist()
probs = np.array(probs)

df = pd.DataFrame({'id': test_df['id'], 'prediction': [ID2LABEL[p] for p in probs.argmax(1)]})
df.to_csv('submission_hizli_large.csv', index=False)
print('\n✅ submission_hizli_large.csv HAZIR!')
